# 项目V1.0：GridSearchCV 调参 —— 第3天任务单
日期：2026-08-19 周三（第39天）

## 今日目标
- 1. 解决昨天的痛点：轻微类召回率只有 0.17
- 2. 用 GridSearchCV 自动搜索逻辑回归的最优超参数（C、class_weight）
- 3. 对比调参前后，确定最终模型（明天基于它输出完整评估报告）

## 一、回顾当前问题

昨天逻辑回归在测试集上的分类报告：

| 类别 | 召回率 | 说明 |
|------|--------|------|
| 一般 | 1.00 | ✅ |
| 严重 | 1.00 | ✅ |
| 致命 | 1.00 | ✅ |
| 轻微 | 0.17 | ❌ 6 个只找到 1 个，漏了 5 个 |

根因有两个（都要记住）：

1. **轻微类样本太少**：500 条里只有 19 条轻微（3.8%）。训练集里只有 13 条（不是 30），测试集 6 条。模型"见过"的轻微太少，学不会它的样子。
2. **模型太"佛系"**：默认 `C=1` 时正则化偏强，模型不敢用力贴合训练数据，轻微/一般的分界线放歪了。

**今日目标：在不明显降低整体准确率的前提下，把轻微类召回率拉起来。**

## 二、调参策略（大白话）

| 参数 | 大白话解释 | 候选值 |
|------|-----------|--------|
| C | 正则化强度的"反义词"：C 越大，模型越敢用力学训练数据；C 越小，模型越"佛系" | 0.01 / 0.1 / 1 / 10 / 100 |
| class_weight | 给类别"发配重"：None 一视同仁；balanced 自动给少数类更高权重 | None / balanced |
| solver | 求解器：lbfgs 支持多分类；liblinear 只支持二分类 | lbfgs |

**GridSearchCV 是什么？** 一个"参数大扫荡机器人"：你把候选值告诉它，它把所有组合（5×2×1=10 种）都用 5 折交叉验证跑一遍，自动选出分数最高的组合。

⚠️ **先记住一个坑：liblinear 不能用于 4 分类**

原计划想试 `solver=["lbfgs", "liblinear"]`。但 liblinear 是 C++ 写的经典求解器，内部只实现了二分类算法，不支持 3 个及以上类别。直接放网格里会报错。等下第 1 步代码里你会亲眼看到这个报错——这也是面试/工作中很常见的报错，值得认识。

## 三、开始干活
### 第0步：准备数据（内核重启过才需要）

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. 读数据
df = pd.read_csv("defect_data.csv")

# 2. 标签编码：严重程度文字 → 数字（一般=0, 严重=1, 致命=2, 轻微=3）
label_encoder = LabelEncoder()
df["严重程度_编码"] = label_encoder.fit_transform(df["严重程度"])

# 3. 特征OneHot编码：缺陷类型、复现概率 → 0/1列
#    drop_first=True 避免"虚拟变量陷阱"（类别列完全线性相关）
df_encoded = pd.get_dummies(
    df, columns=["缺陷类型", "复现概率"], prefix=["类型", "复现"], drop_first=True
)

# 4. 分离特征X和标签y
X = df_encoded.drop(["严重程度", "严重程度_编码"], axis=1)
y = df_encoded["严重程度_编码"]

# 5. 划分训练/测试（stratify=y：分层抽样，保证各类别比例一致）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 6. 标准化（scaler只用训练集fit，测试集只transform——防止数据泄露）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("数据准备完成")
print(f"训练集形状: {X_train_scaled.shape}")
print(f"测试集形状: {X_test_scaled.shape}")
print(f"标签类别: {label_encoder.classes_.tolist()}")

数据准备完成
训练集形状: (350, 6)
测试集形状: (150, 6)
标签类别: ['一般', '严重', '致命', '轻微']


#### 逐段讲解：

1. **`LabelEncoder()`**：把文字标签变成数字。注意排序是自动的（一般=0、严重=1、致命=2、轻微=3）。等下看分类报告时，类别名从 `label_encoder.classes_` 取，不会对错号。

2. **`get_dummies(..., drop_first=True)`**：比如"缺陷类型"有 功能/性能/界面 3 类，只生成 2 列。为什么？3 列会完全线性相关（知道前 2 列就能推出第 3 列），模型会"懵"。

3. **`stratify=y`**：分层抽样。保证 350 条训练集里各类别比例和全量 500 条一致——否则随机切可能把轻微全切到测试集。

4. **`scaler` 只用训练集 fit**：如果对全量数据 fit 再切，测试集信息就"泄露"进训练过程了，评估就不准了。这个叫数据泄露，面试常问。

### 第1步：亲眼看 liblinear 的报错（1分钟）

In [9]:
from sklearn.linear_model import LogisticRegression

# 故意用 liblinear 训练 4 分类，看看会发生什么
try:
    m = LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)
    m.fit(X_train_scaled, y_train)
    print("居然成功了？")
except ValueError as e:
    print("🚨 报错了！报错内容是：")
    print(e)

🚨 报错了！报错内容是：
The 'liblinear' solver does not support multiclass classification (n_classes >= 3). Either use another solver or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.


**这段报错怎么读（以后见到秒懂）：**

- `does not support multiclass classification (n_classes >= 3)` = "不支持 3 个及以上类别的多分类"

- 后半句给了两条出路："换一个求解器" 或 "用 OneVsRestClassifier 包一层"

- 我们选择换求解器，用 `lbfgs`（中小数据集首选）

**代码讲解：** `try ... except ValueError as e` 的意思是"先试试，出错别崩，把错误信息存进 `e` 打印出来"。没有这层保护，程序会直接报红中断。

### 第2步：定义参数网格，跑 GridSearchCV（今天的主角）

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# ========= 1.定义网格参数 =========
# 字典: key 是参数名, value是要试的候选值列表
# 组合总数 = 5(C) * 2(class_weight) * 1(solver) = 10 种

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],  # 正则化强度: 越大越敢学
    "class_weight": [None, "balanced"],  # None一视同仁; balanced 给少数类加权
    "solver": ["lbfgs"],  # 多分类求解器 (liblinear不行, 第1步已验证)
}

# ========== 2.创建 GridSearchCV ==========
# base_model: 模型"骨架",具体参数由网格来填
base_model = LogisticRegression(max_iter=1000, random_state=42)

# estimator : 要调参的模型
# param_grid: 候选参数组合
# cv=5      : 5折交叉验证（每个组合都要被交叉验证打分）
# scoring   : 用什么指标选最优（"accuracy"；想更照顾少数类可换"f1_macro"）
# n_jobs=-1 : 用满所有CPU核并行加速；如果Windows报PermissionError就改成1
grid = GridSearchCV(base_model, param_grid, cv=5, scoring="accuracy", n_jobs=-1)

# ===== 3. 开始"大扫荡" =====
# 内部逻辑：10种组合 × 5折 = 50次训练，全跑完
grid.fit(X_train_scaled, y_train)

# ===== 4. 看结果 =====
print("===== GridSearchCV 调参结果 =====")
print(f"最优参数：{grid.best_params_}")
print(f"最优交叉验证分数：{grid.best_score_:.4f}")

===== GridSearchCV 调参结果 =====
最优参数：{'C': 10, 'class_weight': None, 'solver': 'lbfgs'}
最优交叉验证分数：0.9971


**重点（和直觉相反，好好想想）**

最优参数里 `class_weight` 是 `None`，不是 `balanced`！  
说明在这份数据上，提升轻微类召回率的关键不是"给轻微加权"，而是 `C=10`（模型终于敢用力学）。原因见第四节。

**代码讲解:**

- `GridSearchCV` 的 `cv=5`：把训练集切成 5 份，轮流拿 4 份训练、1 份验证，5 次取平均——防止"运气好"的单次划分骗了你。
- `best_params_`：分数最高的参数组合；`best_score_`：它对应的交叉验证平均分。
- `n_jobs=-1`：并行加速。数据小其实无所谓，但大项目里很关键。Windows 个别环境报 `PermissionError` 就改成 `n_jobs=1`，结果完全一样。

### 第3步：看完整"成绩单"（cv_results_）

In [6]:
import pandas as pd

# 把成绩单转成 DataFrame
results_df = pd.DataFrame(grid.cv_results_)

# 挑几列：参数 + 平均分 + 标准差 + 排名
# mean_test_score : 5折平均准确率
# std_test_score  : 5折标准差（越小越稳定）
# rank_test_score : 排名（1=最优）
show_cols = [
    "param_C",
    "param_class_weight",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
results_df[show_cols].sort_values("rank_test_score")

,param_C,param_class_weight,mean_test_score,std_test_score,rank_test_score
6,10.00,NaN,0.997143,0.005714,1
7,10.00,balanced,0.997143,0.005714,1
9,100.00,balanced,0.997143,0.005714,1
8,100.00,NaN,0.997143,0.005714,1
4,1.00,NaN,0.954286,0.010690,5
5,1.00,balanced,0.945714,0.033074,6
2,0.10,NaN,0.851429,0.057570,7
3,0.10,balanced,0.808571,0.062335,8
1,0.01,balanced,0.734286,0.072054,9
0,0.01,NaN,0.731429,0.082015,10


**能看出什么：**

- 1. C≥10 时，class_weight 用不用都一样（0.9971）——数据完全可分时，加权没有意义。
- 2. C 越小分数越低、标准差越大——模型越"佛系"越学不动，还越不稳定。
- 3. GridSearchCV 在并列第一里选了最先出现的`C=10, None`。

**代码讲解：** `pd.DataFrame(grid.cv_results_)` 把 cv_results_（一个大字典，几十个字段）转成表格好读；`sort_values("rank_test_score") `按排名排序，第一名在最上面。

### 第4步：用最优模型在测试集上出"最终成绩"

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# best_estimator_ = 用最优参数在 [全部训练集] 上重新训练好的模型
best_model = grid.best_estimator_

# 在测试集上预测 (测试集模型从没见过)
y_pred_best = best_model.predict(X_test_scaled)

print("====== 最优模型测试集评估 ======")
print(f"准确率: {accuracy_score(y_test, y_pred_best):.4f}")
print("\n分类报告:")
print(
    classification_report(
        y_test, y_pred_best, target_names=label_encoder.classes_.tolist()
    )
)
print(f"混淆矩阵: \n{confusion_matrix(y_test, y_pred_best)}")

====== 最优模型测试集评估 ======
准确率: 1.0000

分类报告:
              precision    recall  f1-score   support

          一般       1.00      1.00      1.00        47
          严重       1.00      1.00      1.00        62
          致命       1.00      1.00      1.00        35
          轻微       1.00      1.00      1.00         6

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

混淆矩阵: 
[[47  0  0  0]
 [ 0 62  0  0]
 [ 0  0 35  0]
 [ 0  0  0  6]]


**轻微类召回率 0.17 → 1.00！目标达成，整体准确率没有降，反而升到 1.0。**

### 第5步：调参前后对比（对照实验）

In [8]:
from sklearn.metrics import accuracy_score, classification_report

# 调参前的基线模型：就是昨天用的默认参数
model_before = LogisticRegression(max_iter=1000, random_state=42)
model_before.fit(X_train_scaled, y_train)
y_pred_before = model_before.predict(X_test_scaled)

print("===== 调参前（默认参数 C=1） =====")
print(f"准确率：{accuracy_score(y_test, y_pred_before):.4f}")
print(
    classification_report(
        y_test, y_pred_before, target_names=label_encoder.classes_.tolist()
    )
)

print("\n===== 调参后（最优参数 C=10） =====")
print(f"准确率：{accuracy_score(y_test, y_pred_best):.4f}")
print(
    classification_report(
        y_test, y_pred_best, target_names=label_encoder.classes_.tolist()
    )
)

===== 调参前（默认参数 C=1） =====
准确率：0.9667
              precision    recall  f1-score   support

          一般       0.90      1.00      0.95        47
          严重       1.00      1.00      1.00        62
          致命       1.00      1.00      1.00        35
          轻微       1.00      0.17      0.29         6

    accuracy                           0.97       150
   macro avg       0.98      0.79      0.81       150
weighted avg       0.97      0.97      0.96       150


===== 调参后（最优参数 C=10） =====
准确率：1.0000
              precision    recall  f1-score   support

          一般       1.00      1.00      1.00        47
          严重       1.00      1.00      1.00        62
          致命       1.00      1.00      1.00        35
          轻微       1.00      1.00      1.00         6

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



### 第6步：单独看 C 的作用

In [10]:
from sklearn.metrics import recall_score

# 固定 class_weight=None，只改 C，看轻微类召回率怎么变
print("C      准确率     轻微类召回率")
for C in [0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(C=C, solver="lbfgs", max_iter=1000, random_state=42)
    m.fit(X_train_scaled, y_train)  # 同一个训练集
    p = m.predict(X_test_scaled)  # 同一个测试集
    acc = accuracy_score(y_test, p)
    # labels=[3]只统计"轻微"这一类；average=None返回每类召回率数组，取第1个
    recall_minor = recall_score(y_test, p, labels=[3], average=None)[0]
    print(f"{C:<7} {acc:.4f}     {recall_minor:.4f}")

C      准确率     轻微类召回率
0.01    0.7667     0.0000
0.1     0.8867     0.0000
1       0.9667     0.1667
10      1.0000     1.0000
100     1.0000     1.0000


一眼就看明白：**C 到 10 是拐点**。之前不是"轻微学不会"，是"模型没敢用力学"。

## 四、为什么 C=10 就全对了？

C 是正则化的"反义词"：C 小 = 强正则化 = 模型怕过拟合、不敢学太细；C 大 = 弱正则化 = 模型放开手脚贴合数据。

你的数据是规则生成的：严重程度 = 特征加权打分，特征一确定，标签就唯一确定 → 完全可分、零噪声。

这种数据下，模型只要够"用力"（C≥10），就能把打分规则完整学出来，测试集自然全对。

所以真正的问题是欠拟合（没学够），不是类不平衡——这就是为什么 `class_weight` 不是关键。

### 三个诚实提醒（写简历 / 面试）

- **100% 准确率不稀奇**：模拟数据 + 规则标签的必然结果。简历必须写"500 条模拟数据"，不能吹成真实政务数据。
- **排查顺序**：先调 C（拟合能力）→ 再调 `class_weight`（类别权重）→ 最后才考虑加样本。这个顺序面试官爱听。
- **真实数据有噪声**，C 太大会过拟合。真实场景要靠交叉验证选"分数高且稳定"的 C，而不是无脑 100。

## 五、决策标准

| 情况 | 表现 | 选择 |
|------|------|------|
| 轻微召回明显提升 + 准确率略降但 >0.92 | 权衡成功 | ✅ 采用调参后模型 |
| 轻微召回提升不大 + 准确率明显下降 | 权衡失败 | 保留原模型，如实记录 |

本次属于**第一种情况（且更极端：准确率不降反升）**：采用 `C=10, class_weight=None` 的模型，作为明天的最终评估对象。

## 2026-08-19 周三（第39天）项目V1.0：GridSearchCV调参

### 调参目标
- 提升轻微类召回率（当前0.17）
- 保持严重/致命类召回率为1.00
- 整体准确率不低于0.92

### 参数网格
- C: [0.01, 0.1, 1, 10, 100]
- class_weight: [None, "balanced"]
- solver: ["lbfgs"]（liblinear不支持多分类，已踩坑验证）

### 调参结果
- 最优参数：__{'C': 10, 'class_weight': None, 'solver': 'lbfgs'}__
- 最优交叉验证分数：__0.9971__
- 调参后测试集准确率：__1.0000__

### 调参前后对比
| 指标 | 调参前（C=1，默认） | 调参后（C=10） | 变化 |
|------|------|------|------|
| 测试集准确率 | 0.967 | 1.000 | ✅ 提升 |
| 轻微类召回率 | 0.17 | 1.00 | ✅ 大幅提升 |
| 严重类召回率 | 1.00 | 1.00 | ✅ 保持 |
| 致命类召回率 | 1.00 | 1.00 | ✅ 保持 |
| 今日目标 | — | 轻微召回↑ + 准确率≥0.92 | ✅ 达成 |

### 关键发现
- C 是这组数据的关键（10 是拐点），class_weight 不是
- 根因是"欠拟合（没敢用力学）"，不是类不平衡
- 100% 准确率是模拟数据的特性，简历如实写

### 最终决策
- 采用哪个模型？__采用调参后模型（C=10）__
- 理由：__轻微召回大幅提升且准确率不降反升__

### 遇到的问题
- __liblinear 报错（已解决，换 lbfgs）；若 n_jobs=-1 报 PermissionError，改成 1__